# Análise exploratória — Brasil 1985–2024
Notebook de diagnóstico da camada processada. Os gráficos abaixo servem para checar qualidade, outliers, quebras e normalizações; não são o dashboard final.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..')
PANEL = ROOT / 'data' / 'processed' / 'painel_governos.parquet'
df = pd.read_parquet(PANEL)
df['data'] = pd.to_datetime(df['data'])
df.shape, df.columns.tolist()


## Cobertura por indicador
Primeiro diagnóstico: início/fim, número de observações e número de governos cobertos.

In [ ]:
coverage = (df.groupby('indicador')
              .agg(inicio=('data','min'), fim=('data','max'), n=('valor','count'), governos=('governo','nunique'))
              .sort_values('inicio'))
coverage


## Série bruta vs. normalizada
Escolha um indicador monetário deflacionado ou um indicador com base 100.

In [ ]:
indicator = 'pib_nominal_trimestral'
g = df[df['indicador'].eq(indicator)].sort_values('data')
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(g['data'], g['valor'], label='valor normalizado')
if 'valor_nominal' in g and g['valor_nominal'].notna().any():
    ax.plot(g['data'], g['valor_nominal'], label='valor nominal', alpha=.65)
ax.set_title(indicator)
ax.legend(); ax.grid(alpha=.2); plt.show()


## Trajetória base 100 por mandato
Base 100 compara trajetória interna, não nível absoluto.

In [ ]:
indicator = 'cambio_venda'
g = df[df['indicador'].eq(indicator)].dropna(subset=['indice_base100'])
fig, ax = plt.subplots(figsize=(12,5))
for gov, part in g.groupby('governo'):
    ax.plot(part['mes_mandato'].astype(float), part['indice_base100'], label=gov)
ax.axhline(100, linewidth=1, linestyle='--')
ax.set_xlabel('Mês do mandato'); ax.set_ylabel('Índice (mês 1 = 100)')
ax.legend(ncol=3, fontsize=8); ax.grid(alpha=.2); plt.show()


## Distribuições e outliers
Use boxplots como diagnóstico; diferenças de nível entre governos não identificam causalidade.

In [ ]:
indicator = 'ipca_mensal'
g = df[df['indicador'].eq(indicator)].dropna(subset=['governo','valor'])
order = g['governo'].drop_duplicates().tolist()
data = [g.loc[g['governo'].eq(k),'valor'].to_numpy() for k in order]
fig, ax = plt.subplots(figsize=(13,5))
ax.boxplot(data, labels=order, showfliers=True)
ax.tick_params(axis='x', rotation=45)
ax.set_title(f'Distribuição: {indicator}'); ax.grid(axis='y', alpha=.2); plt.show()


## Flags metodológicas
Verifique quebras, anos de atribuição ambígua e períodos parciais antes de qualquer teste.

In [ ]:
flag_cols = [c for c in ['quebra_metodologica','atribuicao_governo_ambigua','comparacao_direta_valida','governo_em_andamento'] if c in df]
df.groupby(['indicador','governo'])[flag_cols].sum(numeric_only=True).head(50)


## Correlações exploratórias
Correlação não implica causalidade; para séries não estacionárias, considere diferenças/retornos antes de interpretar.

In [ ]:
monthly = (df.assign(mes=df['data'].dt.to_period('M').dt.to_timestamp('M'))
             .pivot_table(index='mes', columns='indicador', values='valor', aggfunc='last'))
corr = monthly.corr()
corr.round(2)
